# 🏋️ Ćwiczenie 7: Klasyfikacja — Rezygnacje klientów siłowni

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 7 (17.11.2026)
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐⭐ (Zaawansowany)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Budować `Pipeline` łączący preprocessing z modelem klasyfikacyjnym
2. Trenować i porównywać `LogisticRegression` oraz `RandomForestClassifier`
3. Czytać confusion matrix i `classification_report`
4. Rysować krzywą ROC i liczyć AUC
5. Stosować k-fold cross-validation do wiarygodnej oceny modelu
6. Stroić hiperparametry przy pomocy `GridSearchCV`
7. Analizować feature importance i kompromis precision/recall (threshold tuning)
8. Radzić sobie z niezbalansowanymi klasami

## 📋 Wymagania wstępne
- Feature Engineering i `Pipeline` (Wykład 6)
- Podstawy `train_test_split` i metryk ewaluacji

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz, sprawdź hint (komórka z `@title`) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score,
)

np.random.seed(42)  # 👈 powtarzalność wyników

print("✅ Setup zakończony! Zaczynamy 🚀")

## 📖 Wprowadzenie — klasyfikacja w pigułce

Dziś budujemy model przewidujący, czy klient siłowni **zrezygnuje z członkostwa** (`churn`). To klasyczny problem **klasyfikacji binarnej**.

| Koncept | Opis | Przykład |
|---------|------|----------|
| `Pipeline` | Łączy preprocessing z modelem w jeden obiekt | `Pipeline([("scaler", ...), ("model", ...)])` |
| Confusion matrix | Macierz TP/TN/FP/FN | `confusion_matrix(y_test, y_pred)` |
| Precision/Recall/F1 | Metryki jakości klasyfikacji | `classification_report(...)` |
| ROC-AUC | Jakość rankingu predykcji niezależna od progu | `roc_auc_score(y_test, probs)` |
| Cross-validation | Wiarygodna ocena modelu na wielu podziałach | `cross_val_score(model, X, y, cv=5)` |
| `GridSearchCV` | Automatyczne strojenie hiperparametrów | `GridSearchCV(model, param_grid, cv=5)` |

Dziś pracujemy na **syntetycznym zbiorze ~6 000 klientów** siłowni fitness 🏋️‍♀️.

In [ ]:
# @title 📊 Generowanie danych — rezygnacje klientów siłowni

N_KLIENTOW = 6000

wiek = np.random.randint(16, 70, N_KLIENTOW)
plec = np.random.choice(["K", "M"], N_KLIENTOW)
dlugosc_czlonkostwa_mies = np.random.randint(1, 60, N_KLIENTOW)
wizyty_miesiecznie = np.clip(np.random.normal(8, 4, N_KLIENTOW), 0, 30).round(1)
ocena_satysfakcji = np.clip(np.random.normal(6.5, 2.0, N_KLIENTOW), 1, 10).round(1)
liczba_zamrozen = np.random.poisson(0.6, N_KLIENTOW)
zajecia_grupowe = np.random.choice([True, False], N_KLIENTOW, p=[0.4, 0.6])

# Prawdopodobieństwo churnu rośnie przy: mało wizyt, niska satysfakcja, dużo zamrożeń
# i maleje przy: długim stażu członkostwa i korzystaniu z zajęć grupowych
logit = (
    -1.5
    - 0.15 * wizyty_miesiecznie
    - 0.35 * ocena_satysfakcji
    + 0.55 * liczba_zamrozen
    - 0.03 * dlugosc_czlonkostwa_mies
    - 0.5 * zajecia_grupowe.astype(int)
    + np.random.normal(0, 1.0, N_KLIENTOW)  # 👈 szum
)
prawdopodobienstwo_churn = 1 / (1 + np.exp(-logit))
churn = (np.random.uniform(0, 1, N_KLIENTOW) < prawdopodobienstwo_churn).astype(int)

df = pd.DataFrame({
    "wiek": wiek,
    "plec": plec,
    "dlugosc_czlonkostwa_mies": dlugosc_czlonkostwa_mies,
    "wizyty_miesiecznie": wizyty_miesiecznie,
    "ocena_satysfakcji": ocena_satysfakcji,
    "liczba_zamrozen": liczba_zamrozen,
    "zajecia_grupowe": zajecia_grupowe,
    "churn": churn,
})

print(f"📊 Liczba klientów: {len(df):,}")
print(f"📉 Procent rezygnacji (churn): {df['churn'].mean():.1%}")
df.head()

## 🎓 Przykład prowadzony — baseline Logistic Regression

**Zadanie:** Zbudujemy `Pipeline` z preprocessingiem (skalowanie cech numerycznych + encoding `plec`) i `LogisticRegression`, a następnie ocenimy go przy pomocy confusion matrix i `classification_report`.

In [ ]:
# @title 🎓 Przykład prowadzony — uruchom i przeanalizuj

X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cechy_numeryczne = [
    "wiek", "dlugosc_czlonkostwa_mies", "wizyty_miesiecznie",
    "ocena_satysfakcji", "liczba_zamrozen",
]
cechy_kategoryczne = ["plec", "zajecia_grupowe"]

preprocessing = ColumnTransformer([
    ("num", StandardScaler(), cechy_numeryczne),
    ("cat", OneHotEncoder(drop="if_binary"), cechy_kategoryczne),
])

baseline_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_baseline))
print()
print(classification_report(y_test, y_pred_baseline))

---
## ✏️ Ćwiczenie 1: Random Forest — trening i porównanie z baseline

**Trudność:** ⭐⭐

Zbuduj **nowy** `Pipeline` z tym samym preprocessingiem, ale z `RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)` zamiast Logistic Regression. Wytrenuj go na tym samym `X_train`/`y_train` i porównaj `accuracy`, `f1` z baseline.

> 💡 **Hint:** Możesz ponownie użyć obiektu `preprocessing` — `ColumnTransformer` da się użyć w wielu pipeline'ach.

In [ ]:
# ✏️ Ćwiczenie 1: Random Forest

# TODO: zbuduj Pipeline z preprocessingiem i RandomForestClassifier
rf_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", ...),  # TODO: RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
])

# TODO: wytrenuj pipeline na X_train, y_train
...

# TODO: wygeneruj predykcje na X_test
y_pred_rf = ...

accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"Baseline (Logistic Regression) — F1: {f1_score(y_test, y_pred_baseline):.3f}")
print(f"Random Forest — Accuracy: {accuracy_rf:.3f}, F1: {f1_rf:.3f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: ten sam preprocessing co w baseline, ale model to RandomForestClassifier
rf_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)),
])

# Krok 2: trenujemy na tym samym podziale train/test co baseline, żeby porównanie było uczciwe
rf_pipeline.fit(X_train, y_train)

# Krok 3: predykcje na zbiorze testowym
y_pred_rf = rf_pipeline.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"Baseline (Logistic Regression) — F1: {f1_score(y_test, y_pred_baseline):.3f}")
print(f"Random Forest — Accuracy: {accuracy_rf:.3f}, F1: {f1_rf:.3f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    assert isinstance(rf_pipeline.named_steps["model"], RandomForestClassifier), "❌ Model powinien być RandomForestClassifier"
    assert len(y_pred_rf) == len(y_test), "❌ Liczba predykcji powinna równać się liczbie próbek testowych"
    assert 0.0 <= accuracy_rf <= 1.0, "❌ Accuracy powinno być w zakresie [0, 1]"
    print("✅ Świetnie! Random Forest wytrenowany i porównany z baseline 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 2: Krzywa ROC i AUC

**Trudność:** ⭐⭐⭐

Dla modelu Random Forest z Ćwiczenia 1 policz prawdopodobieństwa (`predict_proba`), narysuj krzywą ROC i policz `roc_auc_score`.

**Oczekiwany wynik:** Wykres z krzywą ROC oraz linią odniesienia (losowy model), a w tytule wartość AUC.

In [ ]:
# ✏️ Ćwiczenie 2: ROC curve i AUC

# TODO: policz prawdopodobieństwa klasy pozytywnej (churn=1) dla X_test
probs_rf = ...  # TODO: rf_pipeline.predict_proba(X_test)[:, 1]

# TODO: policz fpr, tpr, thresholds za pomocą roc_curve
fpr, tpr, thresholds = ...

# TODO: policz AUC
auc_rf = ...

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_rf:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Losowy model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Krzywa ROC — Random Forest")
plt.legend()
plt.show()

print(f"AUC: {auc_rf:.3f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: predict_proba zwraca macierz [P(klasa=0), P(klasa=1)] — bierzemy kolumnę klasy pozytywnej (churn=1)
probs_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Krok 2: roc_curve liczy FPR/TPR dla wielu progów decyzyjnych na podstawie prawdopodobieństw
fpr, tpr, thresholds = roc_curve(y_test, probs_rf)

# Krok 3: AUC to pole pod krzywą ROC — im bliżej 1.0, tym lepszy ranking predykcji
auc_rf = roc_auc_score(y_test, probs_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_rf:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Losowy model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Krzywa ROC — Random Forest")
plt.legend()
plt.show()

print(f"AUC: {auc_rf:.3f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    assert len(probs_rf) == len(y_test), "❌ Liczba prawdopodobieństw powinna równać się liczbie próbek testowych"
    assert 0.5 <= auc_rf <= 1.0, f"❌ AUC powinno być między 0.5 a 1.0, jest {auc_rf}"
    print("✅ Krzywa ROC narysowana, AUC policzone poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 3: K-fold cross-validation

**Trudność:** ⭐⭐⭐

Wykonaj **5-fold cross-validation** dla `rf_pipeline` na **całym** zbiorze `X`/`y` (nie tylko train!), używając `scoring="f1"`. Wypisz wynik dla każdego folda oraz średnią i odchylenie standardowe.

> 💡 **Hint:** `cross_val_score(pipeline, X, y, cv=5, scoring="f1")` sam zadba o dopasowanie preprocessingu osobno w każdym foldzie.

In [ ]:
# ✏️ Ćwiczenie 3: Cross-validation

# TODO: wykonaj 5-fold cross-validation dla rf_pipeline na CAŁYM X, y (scoring="f1")
cv_scores = ...  # TODO: cross_val_score(rf_pipeline, X, y, cv=5, scoring="f1")

print(f"F1 per fold: {cv_scores.round(3)}")
print(f"Średnia: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: cross_val_score samo dzieli X, y na 5 foldów i w każdym dopasowuje preprocessing + model od nowa
# (dzięki temu nie ma wycieku danych testowych do treningu w żadnym foldzie)
cv_scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring="f1")

print(f"F1 per fold: {cv_scores.round(3)}")
print(f"Średnia: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    assert len(cv_scores) == 5, f"❌ Oczekiwano 5 wyników (5-fold), jest {len(cv_scores)}"
    assert all(0.0 <= s <= 1.0 for s in cv_scores), "❌ Wyniki F1 powinny być w zakresie [0, 1]"
    print("✅ Cross-validation wykonana poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 4: Strojenie hiperparametrów z GridSearchCV

**Trudność:** ⭐⭐⭐⭐

Użyj `GridSearchCV` do znalezienia najlepszej kombinacji `n_estimators` i `max_depth` dla `RandomForestClassifier` wewnątrz pipeline'u. Siatka parametrów: `n_estimators: [100, 200, 300]`, `max_depth: [5, 10, 15]`. Użyj `cv=5` i `scoring="f1"`.

> 💡 **Hint:** Nazwy parametrów w `param_grid` przy strojeniu kroku pipeline'u wymagają prefiksu `model__`, np. `"model__n_estimators"`.

In [ ]:
# ✏️ Ćwiczenie 4: GridSearchCV

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15],
}

tuning_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", RandomForestClassifier(random_state=42)),
])

# TODO: zbuduj GridSearchCV z tuning_pipeline, param_grid, cv=5, scoring="f1"
grid_search = ...

# TODO: dopasuj grid_search na X_train, y_train
...

print(f"Najlepsze parametry: {grid_search.best_params_}")
print(f"Najlepszy F1 (cross-val): {grid_search.best_score_:.3f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: GridSearchCV przeszuka każdą kombinację param_grid (3 x 3 = 9 kombinacji) z 5-fold CV każda
grid_search = GridSearchCV(tuning_pipeline, param_grid, cv=5, scoring="f1")

# Krok 2: fit() trenuje 9 * 5 = 45 modeli i wybiera kombinację z najlepszym średnim F1
grid_search.fit(X_train, y_train)

print(f"Najlepsze parametry: {grid_search.best_params_}")
print(f"Najlepszy F1 (cross-val): {grid_search.best_score_:.3f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    assert isinstance(grid_search, GridSearchCV), "❌ grid_search powinien być instancją GridSearchCV"
    assert hasattr(grid_search, "best_params_"), "❌ Czy na pewno wywołałeś grid_search.fit(...)?"
    assert "model__n_estimators" in grid_search.best_params_, "❌ Brakuje 'model__n_estimators' w best_params_"
    print("✅ GridSearchCV wykonany poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 5: Feature importance

**Trudność:** ⭐⭐

Wyciągnij `feature_importances_` z najlepszego modelu znalezionego przez `GridSearchCV` (`grid_search.best_estimator_`) i narysuj poziomy wykres słupkowy posortowany malejąco.

> 💡 **Hint:** Nazwy cech po `ColumnTransformer` możesz pobrać metodą `.get_feature_names_out()` na kroku `preprocessing` najlepszego pipeline'u.

In [ ]:
# ✏️ Ćwiczenie 5: Feature importance

najlepszy_model = grid_search.best_estimator_.named_steps["model"]
nazwy_cech = grid_search.best_estimator_.named_steps["preprocessing"].get_feature_names_out()

# TODO: stwórz pd.Series z feature_importances_ i nazwami cech jako indeks, posortuj malejąco
waznosc_cech = ...  # TODO: pd.Series(najlepszy_model.feature_importances_, index=nazwy_cech).sort_values(ascending=False)

waznosc_cech.plot(kind="barh", figsize=(7, 5))
plt.title("Feature importance — Random Forest")
plt.gca().invert_yaxis()
plt.show()

print(waznosc_cech)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: feature_importances_ to tablica NumPy w tej samej kolejności co nazwy_cech po preprocessingu
# budujemy z nich pd.Series, żeby móc łatwo sortować i podpisać oś wykresu
waznosc_cech = pd.Series(
    najlepszy_model.feature_importances_, index=nazwy_cech
).sort_values(ascending=False)

waznosc_cech.plot(kind="barh", figsize=(7, 5))
plt.title("Feature importance — Random Forest")
plt.gca().invert_yaxis()
plt.show()

print(waznosc_cech)

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    assert isinstance(waznosc_cech, pd.Series), "❌ waznosc_cech powinno być pd.Series"
    assert len(waznosc_cech) == len(nazwy_cech), "❌ Liczba cech w waznosc_cech nie zgadza się z liczbą cech po preprocessing"
    assert waznosc_cech.is_monotonic_decreasing, "❌ waznosc_cech powinno być posortowane malejąco"
    print("✅ Feature importance policzone i posortowane poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 6: Threshold tuning — kompromis precision/recall

**Trudność:** ⭐⭐⭐

Zamiast domyślnego progu 0.5, sprawdź, co się stanie przy progu **0.3**. Policz `precision` i `recall` dla obu progów i porównaj je w krótkim komentarzu (Markdown, 2-3 zdania): dlaczego siłownia może preferować niższy próg?

**Oczekiwany wynik:** Przy niższym progu `recall` powinien wzrosnąć, a `precision` zwykle spadnie.

In [ ]:
# ✏️ Ćwiczenie 6: Threshold tuning

probs_best = grid_search.best_estimator_.predict_proba(X_test)[:, 1]

# TODO: predykcje przy progu domyślnym 0.5
y_pred_prog_05 = ...  # TODO: (probs_best >= 0.5).astype(int)

# TODO: predykcje przy niższym progu 0.3
y_pred_prog_03 = ...  # TODO: (probs_best >= 0.3).astype(int)

precision_05, recall_05 = precision_score(y_test, y_pred_prog_05), recall_score(y_test, y_pred_prog_05)
precision_03, recall_03 = precision_score(y_test, y_pred_prog_03), recall_score(y_test, y_pred_prog_03)

print(f"Próg 0.5 — precision: {precision_05:.3f}, recall: {recall_05:.3f}")
print(f"Próg 0.3 — precision: {precision_03:.3f}, recall: {recall_03:.3f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

probs_best = grid_search.best_estimator_.predict_proba(X_test)[:, 1]

# Krok 1: przy progu 0.5 klasyfikujemy jako churn tylko gdy model jest "bardziej niż pewny"
y_pred_prog_05 = (probs_best >= 0.5).astype(int)

# Krok 2: obniżając próg do 0.3, łapiemy więcej prawdziwych przypadków churnu (wyższy recall),
# kosztem tego, że więcej osób zostanie błędnie oznaczonych jako zagrożone (niższy precision)
y_pred_prog_03 = (probs_best >= 0.3).astype(int)

precision_05, recall_05 = precision_score(y_test, y_pred_prog_05), recall_score(y_test, y_pred_prog_05)
precision_03, recall_03 = precision_score(y_test, y_pred_prog_03), recall_score(y_test, y_pred_prog_03)

print(f"Próg 0.5 — precision: {precision_05:.3f}, recall: {recall_05:.3f}")
print(f"Próg 0.3 — precision: {precision_03:.3f}, recall: {recall_03:.3f}")

**Twój komentarz (uzupełnij):** _TODO: napisz 2-3 zdania, dlaczego siłownia może preferować niższy próg decyzyjny mimo niższego precision._

> 💡 **Przykładowa odpowiedź:** Dla siłowni koszt **nieprzewidzenia** rezygnacji (false negative) jest zwykle wyższy niż koszt **fałszywego alarmu** (false positive) — niezadowolony, ale pominięty klient po prostu odejdzie bez interwencji, podczas gdy skontaktowanie się z klientem, który i tak by został, kosztuje tylko trochę czasu działu obsługi. Dlatego niższy próg (wyższy recall, niższy precision) jest tu biznesowo korzystny: wolimy zaprosić na rozmowę retencyjną zbyt wielu klientów niż przegapić tych, którzy naprawdę odejdą.

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    assert recall_03 >= recall_05, "❌ Recall przy niższym progu (0.3) powinien być >= recall przy progu 0.5"
    print("✅ Threshold tuning zrozumiany poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## 🏆 Bonus Challenge (opcjonalnie): Niezbalansowane klasy

Sprawdź, jaki jest procent klientów z `churn=1` w Twoim zbiorze. Wytrenuj `RandomForestClassifier(class_weight="balanced", random_state=42)` wewnątrz pipeline'u i porównaj `recall`/`precision` z modelem bez `class_weight`. Czy `class_weight="balanced"` poprawił recall kosztem precision?

In [ ]:
# 🏆 Bonus: class_weight="balanced"

print(f"Procent churn w zbiorze: {y.mean():.1%}")

balanced_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight="balanced", random_state=42
    )),
])

# TODO: wytrenuj balanced_pipeline i porównaj recall/precision z rf_pipeline z Ćwiczenia 1
...

In [ ]:
# @title 💡 Pełne rozwiązanie — Bonus (kliknij, aby zobaczyć krok po kroku)

print(f"Procent churn w zbiorze: {y.mean():.1%}")

# Krok 1: trenujemy wariant z class_weight="balanced" — model "płaci więcej" za pomyłki na rzadszej klasie
balanced_pipeline.fit(X_train, y_train)
y_pred_balanced = balanced_pipeline.predict(X_test)

precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)

precision_rf1 = precision_score(y_test, y_pred_rf)
recall_rf1 = recall_score(y_test, y_pred_rf)

print(f"Bez class_weight     — precision: {precision_rf1:.3f}, recall: {recall_rf1:.3f}")
print(f"class_weight=balanced — precision: {precision_balanced:.3f}, recall: {recall_balanced:.3f}")

if recall_balanced > recall_rf1:
    print("🔍 class_weight='balanced' poprawił recall — model lepiej wyłapuje faktyczne rezygnacje,")
    print("   zwykle kosztem precision (więcej fałszywych alarmów).")

---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Budowę `Pipeline` z preprocessingiem i modelem klasyfikacyjnym
- ✅ Trening i porównanie `LogisticRegression` oraz `RandomForestClassifier`
- ✅ Confusion matrix, `classification_report`, krzywą ROC i AUC
- ✅ K-fold cross-validation i strojenie hiperparametrów przez `GridSearchCV`
- ✅ Feature importance i threshold tuning (kompromis precision/recall)
- ✅ Radzenie sobie z niezbalansowanymi klasami (`class_weight`)

### 📚 Dalsze czytanie
- [scikit-learn: Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn: GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

### 🔮 Next Up
Na **Wykładzie 8 (24.11.2026)** przechodzimy do **Spark MLlib** — ten sam pipeline, ale na danych, które nie mieszczą się na jednym komputerze! 🚀